# Statistik-Labor Testat – NANUAG-Gesamtprojekt (Version 1)
**Wintersemester 2026/2027 – Gruppe 02**

***Hinweise:***
- Die Variablen unten sind bereits auf **Ihre Gruppe (02)** voreingestellt.
- Bitte überprüfen Sie Ihre Resultate laufend mit `Teilergebnisse_G02_V01.txt`.
- **Es wird dreimal abgegeben**, jeweils zusammen mit der zugehörigen Präsentation:
  Zwischenabgabe 1, Zwischenabgabe 2 und Endabgabe. Jede Abgabe besteht aus diesem (wachsenden)
  Notebook **und** einem PDF-Export davon, in Moodle.
  Dateiname je Abgabe: `Projekt_Gruppe02_<Nachnamen>_Zwischenabgabe1.ipynb`
  bzw. `..._Zwischenabgabe2.ipynb` bzw. `..._Endabgabe.ipynb`.
- Konkrete Wochentage/Uhrzeiten werden noch bekanntgegeben; siehe Terminplan für die Kalenderwochen.
- Geben Sie **niemals** die vollständige Tabelle mit 100.000 Zeilen aus — nutzen Sie `.head()`,
  `.sample()`, Aggregationen und Diagramme.
- Die vollständige Aufgabenstellung mit allen Teilaufgaben finden Sie in
  `Aufgabenstellung_Gruppe02.pdf`. Dieses Notebook ist das Arbeitsgerüst.

In [136]:
GRUPPENNUMMER = 2   # bereits auf Ihre Gruppe voreingestellt
VERSION = 1         # dieses Semester fuer alle Gruppen: Version 1

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

g = f"{GRUPPENNUMMER:02d}"
v = f"{VERSION:02d}"
pfad = "./"
print("Arbeite mit Dateien aus:", pfad)

Arbeite mit Dateien aus: Gruppe02/


---
# Teil 1 – Personaldaten der NANUAG

## a) Daten einlesen

In [137]:
roh= pd.read_csv("nanuag_mitarbeiter_G02_V01.csv",sep=";")
daten = roh.copy()
daten.head()

,Personalnummer,Anrede,Vorname,Nachname,Geburtsdatum,Abteilung,Standort,Jahresgehalt,Fuehrungsposition
0,100000,Herr,Peter,Möller,20.03.1999,Produktion,Bremen,39313.52,Nein
1,100001,Frau,Jana,Lang,11.08.1965,Produktion,Bremen,49632.19,Nein
2,100002,Herr,Dirk,Meier,18.02.1998,Produktion,Leipzig,38629.65,Nein
3,100003,Frau,Anja,Müller,07.02.2006,Finanzen,Bremen,48019.24,Nein
4,100004,Divers,Nadine,Braun,04.04.1965,Produktion,Bremen,50169.48,Nein


## b) Datenbereinigung
Dokumentieren Sie Ihre Regeln in einer Tabelle (z. B. als DataFrame).

In [138]:
#Löschen Falscher Geburtsdatum Formate
daten["Geburtsdatum"] = pd.to_datetime(daten["Geburtsdatum"], format="%d.%m.%Y", errors="coerce") #Felder in Geburtsdatum mit einem falschen Format werden mit NaN markiert

fehler = daten["Geburtsdatum"].isna()

print(fehler.sum(), "Geburtsdaten welche nicht im Format Tag.Monat.Jahr sind")

#print(daten[fehler])
daten = daten.dropna(subset=["Geburtsdatum"]) #Zeilen wo es mit Nan markierte Felder gibt werden gelöscht

#Löschen von Geburtsdaten welche in der Zukunft liegen oder zu weit in der Vergangenheit
geburtstag_zukunft_vergangenheit = (daten["Geburtsdatum"] > pd.Timestamp.today()) | (daten["Geburtsdatum"] < pd.Timestamp("1900-01-01"))

print(geburtstag_zukunft_vergangenheit.sum(), "Geburtsdaten welche über dem aktuellen Datum sind oder unter 01.01.1900")
#print(daten[geburtstag_zukunft_vergangenheit])

daten = daten[~geburtstag_zukunft_vergangenheit] #~ steht für Negation. Also alle Werte die True waren wurden False

#Löschen Doppelter Personalnummern
anzahl_dopplungen = daten["Personalnummer"].duplicated() #anzahl_dopplungen wird ein series aus True und False. True für alle dopellten Werte außer den ersten

print(anzahl_dopplungen.sum(), "Personalnummern, welche mehr als 1 Mal vorkommen") #ich zähle die anzahl der True Werte

daten = daten.drop_duplicates(subset="Personalnummer") #entfernt alle doppelten Felder außer das erste


# Leerstellen: erst zählen und anzeigen, dann löschen
leerstellen = daten.isna().any(axis=1)

print(leerstellen.sum(), "Zeilen, welche mind. ein leeres Feld haben")
#print(daten[leerstellen])

daten = daten[~leerstellen]

# Unplausible Gehälter (Rahmen: nur negative Zahlen)
neg_gehalt = (daten["Jahresgehalt"]) <=  0

print(neg_gehalt.sum(), "Jahresgehälter mit negativem Vorzeichen")
#print(daten[neg_gehalt])

daten = daten[~neg_gehalt]


#Keine Ahnung wie ich Zufallszeichen-ketten in Namen überprüfen soll

gehalt_bereinigt = daten.copy()
gehalt_bereinigt = gehalt_bereinigt.sort_values("Jahresgehalt")
Q1 = gehalt_bereinigt["Jahresgehalt"].quantile(0.25)
Q3 = gehalt_bereinigt["Jahresgehalt"].quantile(0.75)
IQR = Q3 - Q1
untergrenze = Q1 - 1.5*IQR
obergrenze = Q3 + 1.5*IQR
gehalts_ausreißer = (gehalt_bereinigt["Jahresgehalt"] < untergrenze) | (gehalt_bereinigt["Jahresgehalt"] > obergrenze) 
print(gehalts_ausreißer.sum(), "Ausreißer nach folgender Regel: [Q1 − 1,5 · IQR, Q3 + 1,5 · IQR]")

gehalt_bereinigt = gehalt_bereinigt[~gehalts_ausreißer]




858 Geburtsdaten welche nicht im Format Tag.Monat.Jahr sind
286 Geburtsdaten welche über dem aktuellen Datum sind oder unter 01.01.1900
284 Personalnummern, welche mehr als 1 Mal vorkommen
0 Zeilen, welche mind. ein leeres Feld haben
198 Jahresgehälter mit negativem Vorzeichen
589 Ausreißer nach folgender Regel: [Q1 − 1,5 · IQR, Q3 + 1,5 · IQR]


## c) Alter bestimmen (Stichtag 31.12.2026)

## d) Explorative Analyse (Säulendiagramme, Tortendiagramme)

## e) Kennzahlen (Alter und Jahresgehalt, gesamt / Herr / Frau)

## f) Zusammenhangsanalyse: Pearson- und Spearman-Korrelation Alter/Jahresgehalt

## g) Diskussion: Erste Vermutung zum Gehaltsunterschied
*(Freitext-Antwort als Markdown-Zelle)*

*Ihre Antwort hier...*

---
# Teil 2 – Unternehmensentwicklung

## a) Daten einlesen und Streudiagramm

In [139]:
umsatz = pd.read_csv(pfad + f"nanuag_umsatz_G{g}_V{v}.txt", sep=r'\s+',
                      decimal=',', header=None, names=['Jahr','Umsatz_Mio_EUR'])
umsatz.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Gruppe02/nanuag_umsatz_G02_V01.txt'

## b) Regressionsanalyse (5 Funktionstypen gemäß Anhang A – abhängig von gerader/ungerader Gruppennummer)

## c) Kreuzvalidierung (Training 1994–2018, Test 2019–2026): R²/RMSE je Modell auf Testdaten

## d) Modellwahl und Begründung (basierend auf b und c) + Residuenplot

## e) Diagramm mit optimaler Regressionskurve

## f) Prognose 2030 / 2035, Vergleich mit Polynom 6. Grades, Diskussion Extrapolation

---
# Teil 3 – Firmenlotterie „NANU-Glück“

## a) Erwartungswert und Standardabweichung von $X_{10}$ (vollständige Rechnung)

## b) $X_9, \dots, X_2$ tabellarisch

## c) Optimale Anzahl angekreuzter Zahlen + Diskussion Risiko vs. Erwartungswert

## d) Individuelle Aufgabe

In [ ]:
with open(pfad + f"nanuag_lotterie_frage_G{g}_V{v}.txt", encoding='utf-8') as f:
    print(f.read())

## e) Kombinationsspiel: $X_{10}+X_2$ gleichzeitig — Unabhängigkeitsdiskussion, Erwartungswert und Standardabweichung

---
# Teil 4 – Qualitätskontrolle in der Fertigung

## a) Daten einlesen und Kennzahlen (MR1, MR2, MR3)

In [ ]:
mr1 = pd.read_csv(pfad + f"nanuag_qc_G{g}_V{v}_MR1.txt", sep=r'\s+', decimal=',', header=None, names=['MR1'])
mr2 = pd.read_csv(pfad + f"nanuag_qc_G{g}_V{v}_MR2.txt", sep=r'\s+', decimal=',', header=None, names=['MR2'])
mr3 = pd.read_csv(pfad + f"nanuag_qc_G{g}_V{v}_MR3.txt", sep=r'\s+', decimal=',', header=None, names=['MR3'])
mr1.describe(), mr2.describe(), mr3.describe()

## b) Histogramme (mit begründeter Klassenwahl, z. B. Sturges / Freedman-Diaconis)

## c) Normalverteilung? (Histogramm + Q-Q-Plot oder Schiefe/Kurtosis)

## d) Dichtekurve für normalverteilte Messreihen einzeichnen + Skalierungsfaktor erläutern

## e) Formaler Normalverteilungstest (Shapiro-Wilk oder Kolmogorov-Smirnov)
*Hinweis: `scipy.stats.shapiro` ist für n > 5000 nicht mehr zuverlässig — ziehen Sie eine
Zufallsstichprobe (z. B. n=2000) und diskutieren Sie „statistische vs. praktische Signifikanz“
bei sehr großen Stichproben.*

---
# Teil 5 – Statistische Tests: Gehalt, Führung und Abteilungszugehörigkeit

## a) Gauß-Test (Einstichproben-Z-Test): MR2 gegen Sollmaß μ0=883,0, bekannt: σ0=1,8

## b) Zweistichproben-t-Test (Jahresgehalt Herr vs. Frau)

## c) Binomialtest (Frauenanteil Führungspositionen, $H_0: p=0{,}30$)

## d) Chi-Quadrat-Unabhängigkeitstest (Kreuztabelle Anrede × Abteilung)

## e) Mehrfachtestproblem: Bonferroni-Korrektur über die vier Tests a)–d) ($\alpha'=\alpha/4$)

## f) Management-Summary (max. 1 Seite)

*Ihre Zusammenfassung hier...*